# Exercise: Implementing a Training Orchestrator

Welcome! In this exercise, you'll build a `MiniTrainer` class from scratch. This class will handle the core logic of a machine learning training loop, just like you might find in popular libraries like 🤗 Transformers or PyTorch Lightning, but simplified to its essentials. This is a crucial skill for understanding and customizing how your models learn.

By the end of this exercise, you will be able to:
*   Implement the core logic for a single training epoch, including the forward pass, loss calculation, backpropagation, and weight updates.
*   Implement a validation loop to evaluate your model's performance on a separate dataset without updating its weights.
*   Orchestrate the entire training process, including running epochs, evaluating the model, and saving the best version.

Let's get started!

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from dataclasses import dataclass
import logging
import pathlib
import shutil

# --- Configuration and Logging Setup ---
# Students do not need to modify this part.

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

@dataclass
class TrainerConfig:
    """Configuration for the MiniTrainer."""
    max_epochs: int = 2
    learning_rate: float = 3e-4
    grad_norm_clip: float = 1.0
    eval_interval: int = 1
    checkpoint_dir: str = "checkpoints_mock"

# --- Mock Components for Testing ---
# Students do not need to modify these components. They are simplified
# versions of a real model and dataloader for this exercise.

class MockModel(nn.Module):
    """A simple mock model for testing purposes."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(10, 20),
            nn.ReLU(),
            nn.Linear(20, 5) # 5 output classes
        )
    def forward(self, x):
        return self.net(x)

def get_mock_dataloader(num_samples=100, batch_size=10, is_train=True) -> DataLoader:
    """Creates a mock dataloader with deterministic data."""
    if is_train:
        torch.manual_seed(42) # Seed for reproducibility
    else:
        torch.manual_seed(1337) # Different seed for validation

    features = torch.randn(num_samples, 10)
    labels = torch.randint(0, 5, (num_samples,))
    dataset = TensorDataset(features, labels)
    return DataLoader(dataset, batch_size=batch_size)

# --- The MiniTrainer Class ---
# This is the class you will be completing.

class MiniTrainer:
    def __init__(self, config: TrainerConfig, model: nn.Module, train_loader: DataLoader, val_loader: DataLoader):
        self.config = config
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader

        self.optimizer = AdamW(model.parameters(), lr=config.learning_rate)
        # A simple scheduler that does nothing, for deterministic testing.
        self.scheduler = LambdaLR(self.optimizer, lr_lambda=lambda epoch: 1.0)
        self.loss_fn = nn.CrossEntropyLoss()

        self.best_val_loss = float('inf')
        self.checkpoints_dir = pathlib.Path(config.checkpoint_dir)
        self.checkpoints_dir.mkdir(exist_ok=True)
        logging.info(f"Initialized MiniTrainer. Checkpoints will be saved in '{self.checkpoints_dir}'.")

    def _save_checkpoint(self, epoch_num: int):
        """Saves the model state."""
        checkpoint_path = self.checkpoints_dir / f"best_model_epoch_{epoch_num}.pt"
        torch.save(self.model.state_dict(), checkpoint_path)
        logging.info(f"Checkpoint saved: {checkpoint_path}")

    def _run_epoch(self, epoch_num: int) -> float:
        """Runs a single training epoch."""
        raise NotImplementedError("You must implement the _run_epoch method!")

    @torch.no_grad()
    def _run_eval(self) -> float:
        """Runs a single evaluation cycle."""
        raise NotImplementedError("You must implement the _run_eval method!")

    def train(self):
        """Orchestrates the full training loop."""
        logging.info("Starting training...")
        for epoch in range(self.config.max_epochs):
            train_loss = self._run_epoch(epoch)
            logging.info(f"[Epoch {epoch+1}/{self.config.max_epochs}] Train Loss: {train_loss:.4f}")

            if (epoch + 1) % self.config.eval_interval == 0:
                val_loss = self._run_eval()
                logging.info(f"[Epoch {epoch+1}/{self.config.max_epochs}] Val Loss: {val_loss:.4f}")
                if val_loss < self.best_val_loss:
                    logging.info(f"New best val loss: {val_loss:.4f}. Saving model...")
                    self.best_val_loss = val_loss
                    self._save_checkpoint(epoch + 1)
        logging.info("Training finished.")

### Exercise 1: Implement the Training Epoch

Your first task is to implement the `_run_epoch` method. This is the heart of the training process where the model learns from the data.

**Your goal:**
Complete the `_run_epoch` method in the `MiniTrainer` class. This method should perform the following steps for each batch of data from the `train_loader`:
1.  **Forward pass:** Get the model's predictions (logits).
2.  **Calculate loss:** Compare the predictions to the true labels.
3.  **Zero gradients:** Clear old gradients before the backward pass.
4.  **Backward pass:** Compute the gradients of the loss with respect to model parameters.
5.  **Clip gradients:** Prevent exploding gradients, a common practice in training large models.
6.  **Update weights:** Adjust the model's weights based on the computed gradients.
7.  **Step the scheduler:** Update the learning rate (though our mock scheduler doesn't do anything, this is best practice).

The method should return the average training loss over all batches in the epoch.

**Hint:** The standard order of operations for a training step is `optimizer.zero_grad()`, `loss.backward()`, and `optimizer.step()`.

In [ ]:
def _run_epoch(self, epoch_num: int) -> float:
    """
    Runs a single training epoch.

    Args:
        epoch_num (int): The current epoch number.

    Returns:
        float: The average training loss for the epoch.
    """
    self.model.train()  # Set the model to training mode
    total_loss = 0.0

    for X_batch, y_batch in self.train_loader:
        ### START CODE HERE ### (~8 lines)
        # 1. Get model predictions (logits)
        logits = self.model(X_batch)

        # 2. Calculate the loss
        loss = self.loss_fn(logits, y_batch)

        # 3. Zero the gradients
        self.optimizer.zero_grad()

        # 4. Perform backpropagation
        loss.backward()

        # 5. Clip gradients to prevent them from exploding
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.grad_norm_clip)

        # 6. Update the weights
        self.optimizer.step()

        # 7. Step the learning rate scheduler
        self.scheduler.step()

        total_loss += loss.item()
        ### END CODE HERE ###

    return total_loss / len(self.train_loader)

# We monkey-patch our implementation onto the class for this exercise
MiniTrainer._run_epoch = _run_epoch

In [ ]:
# --- Test Cell for Exercise 1 ---
print("--- Running Test for Exercise 1: _run_epoch ---")

# Setup
torch.manual_seed(123)
mock_config = TrainerConfig(max_epochs=1)
mock_model = MockModel()
mock_train_loader = get_mock_dataloader(is_train=True)
mock_val_loader = get_mock_dataloader(is_train=False)

# Store initial parameters to check if they have been updated
initial_params = [p.clone() for p in mock_model.parameters()]

# Instantiate and run
trainer = MiniTrainer(mock_config, mock_model, mock_train_loader, mock_val_loader)
train_loss = trainer._run_epoch(0)

# Check 1: Loss should be a plausible float value
assert isinstance(train_loss, float), f"Expected loss to be a float, but got {type(train_loss)}"
expected_loss = 1.6219 # This value is deterministic due to seeds
assert abs(train_loss - expected_loss) < 1e-4, f"Got train_loss={train_loss:.4f}, expected around {expected_loss}"
print(f"✅ Loss check passed. Got {train_loss:.4f}")

# Check 2: Model parameters should have been updated
params_updated = False
for p_initial, p_final in zip(initial_params, mock_model.parameters()):
    if not torch.equal(p_initial, p_final):
        params_updated = True
        break
assert params_updated, "Model parameters were not updated after one epoch!"
print("✅ Parameter update check passed.")

print("\nGreat work! Your `_run_epoch` implementation looks correct.")
# Clean up checkpoint directory
shutil.rmtree(mock_config.checkpoint_dir)

### Exercise 2: Implement the Evaluation Loop

Fantastic! Now that your model can learn, you need a way to evaluate its performance on data it hasn't seen before. This is the purpose of the validation or evaluation loop.

**Your goal:**
Complete the `_run_eval` method. This method is similar to the training loop but simpler. It should:
1.  Iterate over the `val_loader`.
2.  Calculate the loss for each batch.
3.  Aggregate the loss.

Crucially, this process must **not** update the model's weights. You should ensure no gradients are computed, and you must not call the optimizer.

**Hint:** Use the `@torch.no_grad()` decorator (which we've already added for you) and make sure you set the model to evaluation mode with `model.eval()`.

In [ ]:
@torch.no_grad()
def _run_eval(self) -> float:
    """
    Runs a single evaluation cycle.
    The @torch.no_grad() decorator ensures that no gradients are computed.

    Returns:
        float: The average validation loss.
    """
    self.model.eval() # Set the model to evaluation mode
    total_loss = 0.0

    for X_batch, y_batch in self.val_loader:
        ### START CODE HERE ### (~3 lines)
        # 1. Get model predictions (logits)
        logits = self.model(X_batch)

        # 2. Calculate the loss
        loss = self.loss_fn(logits, y_batch)

        total_loss += loss.item()
        ### END CODE HERE ###

    return total_loss / len(self.val_loader)

# Monkey-patch the method onto the class
MiniTrainer._run_eval = _run_eval

In [ ]:
# --- Test Cell for Exercise 2 ---
print("--- Running Test for Exercise 2: _run_eval ---")

# Setup
torch.manual_seed(123)
mock_config = TrainerConfig(max_epochs=1)
mock_model = MockModel()
mock_train_loader = get_mock_dataloader(is_train=True)
mock_val_loader = get_mock_dataloader(is_train=False)

# Store initial parameters to check that they are NOT updated
initial_params = [p.clone() for p in mock_model.parameters()]

# Instantiate and run
trainer = MiniTrainer(mock_config, mock_model, mock_train_loader, mock_val_loader)
val_loss = trainer._run_eval()

# Check 1: Loss should be a plausible float value
assert isinstance(val_loss, float), f"Expected loss to be a float, but got {type(val_loss)}"
expected_loss = 1.6085 # This value is deterministic due to seeds
assert abs(val_loss - expected_loss) < 1e-4, f"Got val_loss={val_loss:.4f}, expected around {expected_loss}"
print(f"✅ Loss check passed. Got {val_loss:.4f}")

# Check 2: Model parameters should NOT have been updated
params_updated = False
for p_initial, p_final in zip(initial_params, mock_model.parameters()):
    if not torch.equal(p_initial, p_final):
        params_updated = True
        break
assert not params_updated, "Model parameters were updated during evaluation, but they should not have been!"
print("✅ Parameter immutability check passed.")

print("\nExcellent! Your `_run_eval` implementation is correct.")
# Clean up checkpoint directory
shutil.rmtree(mock_config.checkpoint_dir)

### Challenge (Optional): Putting It All Together

You have already implemented the two core components of the trainer. The final step is to orchestrate them in the main `train` method. We have provided a skeleton for this method already, but we encourage you to read through it to see how your `_run_epoch` and `_run_eval` methods are called in a complete loop.

There is no code to write for this part, but understanding how the full loop works is key:
- It iterates for `max_epochs`.
- In each epoch, it calls your `_run_epoch` method.
- Periodically (controlled by `eval_interval`), it calls your `_run_eval` method.
- It tracks the best validation loss and calls `_save_checkpoint` when a new best is found.

Let's run the full `train` method to see it all in action!

In [ ]:
# --- Integration Test ---
print("--- Running Integration Test: Full Training Loop ---")

# Setup
torch.manual_seed(123)
config = TrainerConfig(
    max_epochs=3,
    eval_interval=1,
    checkpoint_dir="checkpoints_final_test"
)
model = MockModel()
train_loader = get_mock_dataloader(is_train=True)
val_loader = get_mock_dataloader(is_train=False)

# Instantiate the full trainer
trainer = MiniTrainer(config, model, train_loader, val_loader)

# Store the initial validation loss
initial_val_loss = trainer._run_eval()
print(f"Initial Validation Loss: {initial_val_loss:.4f}")

# Run the full training loop
trainer.train()

# --- Assertions ---
# Check 1: The final best validation loss should be lower than the initial one
final_best_loss = trainer.best_val_loss
assert final_best_loss < initial_val_loss, \
    f"Expected final loss ({final_best_loss}) to be less than initial loss ({initial_val_loss}). Model did not learn."
print(f"\n✅ Learning check passed. Final best loss {final_best_loss:.4f} is lower than initial {initial_val_loss:.4f}.")

# Check 2: A checkpoint file should have been created
checkpoint_dir = pathlib.Path(config.checkpoint_dir)
checkpoints = list(checkpoint_dir.glob("*.pt"))
assert len(checkpoints) > 0, "No checkpoint files were created in the directory."
print(f"✅ Checkpoint creation passed. Found {len(checkpoints)} checkpoint(s).")

# Clean up
shutil.rmtree(config.checkpoint_dir)
print(f"✅ Cleanup passed. Removed '{config.checkpoint_dir}'.")

print("\nCongratulations! You have successfully implemented a complete training orchestrator.")